# 02 · ¿Cuánto cambia el promedio al proteger a cada participante?
**Uso:** retrospectiva local con datos del archivo público. La app no publica la referencia exacta, los parámetros de una persona ni las réplicas del experimento. Los resultados de este notebook no tienen una garantía de publicación DP.

Para abril de 2016, cada persona aporta la media de sus minutos intensos + moderados en días observados. Recortar la contribución a [0,120] es una decisión de diseño pública y reproducible. No se estima el máximo observando el grupo. Se miden por separado el sesgo por recorte y el error adicional por ruido.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'backend/privacy.py').is_file())
sys.path.insert(0, str(ROOT))
RAW = ROOT / 'data/raw'
from backend.experiment import analyze_fitbit
quality, results = analyze_fitbit(RAW/'dailyActivity_merged.csv', period='2016-04', trials=300)
print(quality)
print('ε     MAE (min)   p90 error abs.   % devuelto')
for r in results:
    print(f"{r['epsilon']:3.1f}   {r['mae_minutes']:8.3f}   {r['p90_abs_error']:14.3f}   {r['returned_pct']:10.1f}")

## Mecanismo y presupuesto
Vecindad de **adición/eliminación de una persona completa**. Tras restar el centro público 60, cada valor está en [-60,60]. Añadir o quitar una persona cambia la suma centrada en a lo sumo 60 y el conteo en a lo sumo 1. Se aplican dos mecanismos Laplace independientes: escala 60/(ε/2) y 1/(ε/2). Se suman 60 a la razón ruidosa y se recorta la salida a [0,120]. Si el **conteo ruidoso** es inferior a 8 se omite el valor; se consume igualmente el presupuesto. Tanto el umbral como la razón son posprocesamiento. No se consulta un tamaño privado para decidir si responder.

Composición secuencial: ε/2 + ε/2 = ε, δ = 0. A menor ε, menor influencia admisible de una persona y mayor ruido. La app fija ε=1, máximo acumulado 2 para el conjunto del sistema, y conserva la publicación. El resultado que consultan los usuarios usa aleatoriedad segura del mecanismo de IBM. Solo aquí se usa una semilla de simulación para comparar distribuciones reproducibles. Las 300 réplicas de cada ε **no** serían 300 publicaciones gratuitas sobre clientes reales.

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(6,3))
ax.plot([r['epsilon'] for r in results],[r['mae_minutes'] for r in results],marker='o',label='Error absoluto medio')
ax.plot([r['epsilon'] for r in results],[r['p90_abs_error'] for r in results],marker='o',label='Percentil 90')
ax.axhline(5,color='#747c6c',ls=':',label='Tolerancia propuesta: 5 min')
ax.set(xlabel='Épsilon por publicación',ylabel='Error (min/día)',title='Privacidad y utilidad · Fitbit, abril')
ax.legend();fig.tight_layout();plt.show()
ok=[r['epsilon'] for r in results if r['returned_pct']>=95 and r['mae_minutes'] is not None and r['mae_minutes']<=5]
print('Menor ε ensayado con MAE ≤5 min y ≥95% de resultados:', min(ok) if ok else 'ninguno')

## Interpretación y límites
El criterio de 5 minutos sirve para tendencias grupales del prototipo; no es un umbral clínico ni una norma universal. MAE y p90 se calculan sobre resultados devueltos: interpretar también el porcentaje de omisiones. Una realización puede tener mucho más error que el promedio. Comprobar si otro límite de contribución o una población más pequeña cambiaría la decisión antes de reutilizar parámetros.
    
**Fundamento:** sección sobre media con tamaño privado de S12, lámina 16, y notebook de clase *differentialPrivacyExamples* (consultas acotadas, datasets vecinos). Usamos la implementación Laplace de [Holohan et al., 2019](https://arxiv.org/abs/1907.02444) y [muestreo seguro de Holohan y Braghin, 2021](https://arxiv.org/abs/2107.10138); [NIST SP 800-226](https://doi.org/10.6028/NIST.SP.800-226) orienta la elección de unidad y el análisis del presupuesto. Estas fuentes respaldan el método; ε=1 es nuestra decisión experimental.